# 3장 2강: SDK 개발 환경 구성과 연결 테스트
## 3. Supabase Client 초기화

In [1]:
import os
from dotenv import load_dotenv
from supabase import create_client

# .env 파일의 환경변수 로드
load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

# Client 초기화
# supabase 클라이언트 객체 생성
supabase = create_client(url, key)

print(supabase)

## 4. 연결 테스트와 응답·오류 객체 확인

# 3장 3강: SDK 기반 데이터 생성 구현

## 1. SDK insert 구조

In [4]:
# 사용자 추가
response = supabase.table('users').insert({"email" : "user03@test.org"}).execute()

In [7]:
response.data

[{'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:07.455743'}]

In [38]:
# documents INSERT
response = supabase.table('documents').insert({
    'title' : '두 번째 문서',
    'content' : '본문 내용',
    'user_id' : '1320f352-13eb-44fc-8662-3ae945e616cd'
}).execute()

print(response.data)

[{'id': '53651f2d-0d94-449f-8914-fb6d5dd58aa4', 'title': '두 번째 문서', 'content': '본문 내용', 'user_id': '1320f352-13eb-44fc-8662-3ae945e616cd', 'status': 'draft', 'created_at': '2026-09-09T02:37:46.395818+00:00'}]


In [17]:
# 회원 목록 조회
response = supabase.table('users').select('*').execute()
response.data

[{'id': 'e8057300-87a3-43e2-aee6-e4b0d817d0b2',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:44.799562'},
 {'id': 'bd0f1b0d-5921-4751-9235-3e21dfb722ef',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:22.44246'},
 {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:07.455743'}]

In [ ]:
# ilike : 대소문자 구분없이 like문 실행
response = (supabase.table('users')
            .select('*')
            .ilike('email', 'user04%')
            .single() # 레코드 한 개, 딕셔너리 형태로 반환 
            # single 키워드 사용시에 레코드가 없거나, 1개 이상 존재하는 경우에 APIERROR 발생
            .execute()
)


id


In [25]:

response = (supabase.table('users')
            .select('*')
            .ilike('email', 'user04%')
            .maybe_single() # 레코드가 0 ~ 1 일 때 사용, 레코드가 없다고 예외 발생 안함
            .execute()
)

In [2]:
response = (supabase.table('users')
            .select('*')
            .ilike('email', 'user%')
            .limit(1)
            .offset(1) # 1 인덱스에서 1개 조회 
            .single() 
            .execute()
)

response.data
# {'id': 'bd0f1b0d-5921-4751-9235-3e21dfb722ef',
#  'email': 'user02@test.org',
#  'created_at': '2026-09-09T00:42:22.44246'}

{'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
 'email': 'user03@test.org',
 'created_at': '2026-09-09T01:55:07.455743'}

In [ ]:
# ORDER BY Keyword
response =(
    supabase.table('users')
    .select('*')
    .order('email', desc=True)
    .execute()
)

response.data


[{'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:07.455743'},
 {'id': 'bd0f1b0d-5921-4751-9235-3e21dfb722ef',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:22.44246'},
 {'id': 'e8057300-87a3-43e2-aee6-e4b0d817d0b2',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:44.799562'}]

In [ ]:
# LEFT JOIN

response = (
    supabase.table('documents')
    .select('id, title, content, users(id, email)')
    .execute()
)

response.data



[{'id': 'd865df61-ecaa-4117-90e0-84d5a3726118',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
   'email': 'user03@test.org'}},
 {'id': '53651f2d-0d94-449f-8914-fb6d5dd58aa4',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
   'email': 'user03@test.org'}}]

In [ ]:
# INNER JOIN(!inner)
response = (
    supabase.table('documents')
    .select('id, title, content, users!inner(id, email)')
    .execute()
)
response.data


[{'id': 'd865df61-ecaa-4117-90e0-84d5a3726118',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
   'email': 'user03@test.org'}},
 {'id': '53651f2d-0d94-449f-8914-fb6d5dd58aa4',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
   'email': 'user03@test.org'}}]

In [44]:
# INNER JOIN(!inner)
response = (
    supabase.table('documents')
    .select('id, title, content, users!inner(id, email)')
    .ilike('users.email', 'user%')
    .execute()
)
response.data


[{'id': 'd865df61-ecaa-4117-90e0-84d5a3726118',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
   'email': 'user03@test.org'}},
 {'id': '53651f2d-0d94-449f-8914-fb6d5dd58aa4',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
   'email': 'user03@test.org'}}]

## 2. 입력값과 DB 컬럼 매핑

## 3. 생성 결과와 오류 응답 처리

In [ ]:
create_document(None, "본문 내용입니다.", 1)

# 3장 4강: SDK 기반 데이터 조회 구현

## 1. SDK select 구조

## 2. 조건 필터링: eq, order, limit

## 3. 사용자별 조회와 결과 리스트 처리

# 3장 5강: SDK 기반 데이터 수정·삭제 구현

## 1. SDK update 구조

In [48]:
# update
response = (
    supabase.table('documents')
    .select('*')
    .execute()
)

print(response.data)

[{'id': '53651f2d-0d94-449f-8914-fb6d5dd58aa4', 'title': '두 번째 문서', 'content': '본문 내용', 'user_id': '1320f352-13eb-44fc-8662-3ae945e616cd', 'status': 'draft', 'created_at': '2026-09-09T02:37:46.395818+00:00'}, {'id': 'd865df61-ecaa-4117-90e0-84d5a3726118', 'title': '첫 번째 문서(수정)', 'content': '본문 내용', 'user_id': '1320f352-13eb-44fc-8662-3ae945e616cd', 'status': 'draft', 'created_at': '2026-09-09T01:59:33.905026+00:00'}]


In [ ]:
# UPDATE Keyword
response = (
    supabase.table('documents')
    .update({
        'title' : '첫 번째 문서(수정)'
    })
    .eq('id', 'd865df61-ecaa-4117-90e0-84d5a3726118')
    .execute()
)


## 2. SDK delete 구조

In [52]:
response = (
    supabase.table('users')
    .select('*')
    .execute()
)

response.data

[{'id': 'bd0f1b0d-5921-4751-9235-3e21dfb722ef',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:22.44246'},
 {'id': '1320f352-13eb-44fc-8662-3ae945e616cd',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:07.455743'}]

In [51]:
response = (
    supabase.table('users')
    .delete()
    .eq('id', 'e8057300-87a3-43e2-aee6-e4b0d817d0b2')
    .execute()
)

## 3. 수정·삭제 후 검증과 오류 처리

In [ ]:
# 검증: 상태가 잘 바뀌었는지 다시 조회


# 3장 6강: Supabase Authentication 기본 흐름

## 2. 이메일 기반 회원가입과 로그인

In [ ]:
key = os.getenv('SUPABASE_ANON_KEY')
supabase = create_client(url, key)

"""
supabase.auth : 인증(로그인) / 인가(접근 권한) 관련 메서드를 가지고 있는 객체

테이블 명 : auth.users
"""

# 회원가입
response = supabase.auth.sign_up({
    'email' : 'user01@test.org',
    'password' : 'password1234',
    'phone' : '010-1000-1000'
})

AuthResponse(user=User(id='b8eb5606-3b96-4bd5-a091-8cd3648c5f8a', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 422884, tzinfo=TzInfo(0)), confirmed_at=None, email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 445962, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 453401, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 477735, tzinfo=TzInfo(0)), identities=[UserIdentity(id='b8eb5606-3b96-4bd5-a091-8cd3648c5f8a', identity_id='4ec8a860-c1f2-485b-8088-c3411a95f37f', user_id='b8eb5606-3

In [57]:
# 로그인
response = supabase.auth.sign_in_with_password({
    'email' : 'user01@test.org',
    'password' : 'password1234'
})

response.user

User(id='b8eb5606-3b96-4bd5-a091-8cd3648c5f8a', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 422884, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 445962, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 445962, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 9, 3, 48, 5, 182872, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 9, 3, 48, 5, 185096, tzinfo=TzInfo(0)), identities=[UserIdentity(id='b8eb5606-3b96-4bd5-a091-8cd3648c5f8a', identity_id='4ec8a860-c1f2-48

## 3. 세션, 로그아웃, user 객체

In [63]:
# 현재 로그인된 사용자 확인
user = supabase.auth.get_user() # 현재 로그인한 사용자 정보, 있으면 로그인, None이면 미 인증 상태
if user:
    print(f'로그인 상태 : {user}')
else :
    print(f' 미 로그인 상태 ')

 미 로그인 상태 


In [62]:
# 로그아웃
supabase.auth.sign_out()

## 4. 사용자 ID와 테이블 데이터 연결

# 3장 7강: 사용자별 데이터 접근과 권한 관리

## 2. user_id 기반 필터링

In [ ]:


# 본인 문서만 조회


# 3장 8강: SDK 기반 미니 과제 - 사용자별 문서 CRUD

## 2. Auth와 user_id 연결

In [5]:
import os
from dotenv import load_dotenv
from supabase import create_client

# .env 파일의 환경변수 로드
load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv('SUPABASE_ANON_KEY')

# Client 초기화
# supabase 클라이언트 객체 생성
supabase = create_client(url, key)

print(supabase)

In [10]:
supabase.auth.sign_in_with_password({
    "email" : "user01@test.org",
    "password" : "password1234"
})

user = supabase.auth.get_user().user
user_id = user.id

# 생성: 로그인 사용자의 문서
response = (
    supabase.table("documents")
            .insert({
                "title" : "첫 번째 문서",
                "content" : "본문 내용",
                "user_id" : user_id
            })
            .execute()
)

print(response)

# 조회: 본인 문서만


data=[{'id': '0e7b09ac-3973-40ef-9ef0-3ad25b2d8eb5', 'title': '첫 번째 문서', 'content': '본문 내용', 'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a', 'status': 'draft', 'created_at': '2026-09-10T01:09:27.349363+00:00'}] count=None


In [ ]:
# CRUD Class
class AiLog:

    def __init__(self):
        user = supabase.auth.get_user()
        if not user: # 미로그인 상태
            raise PermissionError('로그인 필요함')

        self.user = user.user

    def create_documents(self, title, content):
        response = (
            supabase.table("documents")
                    .insert({
                        "title" : title,
                        "content" : content,
                        "user_id" : self.user.id
                    }).execute()
        )

        return response.data

    def get_my_documents(self):
        response = (
            supabase.table("documents")
                    .select("*")
                    .eq("user_id", self.user.id)
                    .execute()
        )

        return response.data
ai_log = AiLog()

ai_log.create_documents("세 번째 문서", "본문 내용")
ai_log.get_my_documents()

In [27]:
ai_log = AiLog()

ai_log.create_documents("세 번째 문서", "본문 내용")
ai_log.get_my_documents()
# [{'id': '588d23aa-fd9d-4657-86d4-739f28eff58b',
#   'title': '두 번째 문서',
#   'content': '본문 내용',
#   'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
#   'status': 'draft',
#   'created_at': '2026-09-10T01:21:22.229062+00:00'}]

[{'id': '3ec13f68-d58b-4dbf-814a-bea930eaa699',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:30:00.722286+00:00'}]

In [28]:
ai_log.get_my_documents()

[{'id': '0e7b09ac-3973-40ef-9ef0-3ad25b2d8eb5',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:09:27.349363+00:00'},
 {'id': '588d23aa-fd9d-4657-86d4-739f28eff58b',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:21:22.229062+00:00'},
 {'id': '41144b20-6717-4a52-b42d-8902fc25838e',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:26:47.451612+00:00'},
 {'id': '3ec13f68-d58b-4dbf-814a-bea930eaa699',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:30:00.722286+00:00'}]

In [29]:
# 회원가입
response = supabase.auth.sign_up({
    'email' : 'user02@test.org',
    'password' : 'password1234'
})

In [34]:
supabase.auth.sign_in_with_password({
    'email' : 'user01@test.org',
    'password' : 'password1234'
})

AuthResponse(user=User(id='b8eb5606-3b96-4bd5-a091-8cd3648c5f8a', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 422884, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 445962, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 59, 445962, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 2, 15, 25, 843851, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 2, 15, 25, 848291, tzinfo=TzInfo(0)), identities=[UserIdentity(id='b8eb5606-3b96-4bd5-a091-8cd3648c5f8a', identit

In [36]:
response = (
    supabase.table('documents')
            .select('*')
            .execute()
)

response.data

[{'id': '0e7b09ac-3973-40ef-9ef0-3ad25b2d8eb5',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:09:27.349363+00:00'},
 {'id': '588d23aa-fd9d-4657-86d4-739f28eff58b',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:21:22.229062+00:00'},
 {'id': '41144b20-6717-4a52-b42d-8902fc25838e',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:26:47.451612+00:00'},
 {'id': '3ec13f68-d58b-4dbf-814a-bea930eaa699',
  'title': '세 번째 문서',
  'content': '본문 내용',
  'user_id': 'b8eb5606-3b96-4bd5-a091-8cd3648c5f8a',
  'status': 'draft',
  'created_at': '2026-09-10T01:30:00.722286+00:00'}]

## 3. 응답·오류 처리와 코드 구조 정리